In [ ]:
import os
os.environ["PYTHONUTF8"] = "1"



In [ ]:

pip install "transformers>=4.44.0" "datasets" "accelerate" "bitsandbytes" "peft" "trl"


In [ ]:
# Prepare Translation

from datasets import load_dataset

dataset = load_dataset("opus_books", "en-es")  # just an example
dataset = dataset["train"].train_test_split(test_size=0.05)

def rename_columns(example):
    return {
        "src": example["translation"]["en"],
        "tgt": example["translation"]["es"],
    }

dataset = dataset.map(rename_columns, remove_columns=dataset["train"].column_names)


In [ ]:
def build_prompt(src, tgt=None):
    # tgt is only used for training (label)
    user = f"Translate this from English to Spanish:\n{src}"
    if tgt is None:
        # inference
        return f"User: {user}\nAssistant:"
    else:
        # training
        return f"User: {user}\nAssistant: {tgt}"

In [ ]:
# Load LLama/TinyLlama in 4-bit with QLORA

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import notebook_login

# Try to get the HF_TOKEN from Colab secrets, if not, prompt for login
# try:
#     HF_TOKEN = userdata.get('HF_TOKEN')
#     print("Hugging Face token found in Colab secrets.")
# except userdata.exceptions.KeyNotFoundError:
#     print("HF_TOKEN not found in Colab secrets. Please add it or log in manually.")
#     notebook_login()

# waiting on access to llama 3.2
#model_id = "meta-llama/Llama-3.2-1B-Instruct"
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)


In [ ]:
#Add LoRA Adapters

from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
# Tokenize and format dataset

max_length = 256  # adjust based on your data

def preprocess(example):
    prompt = f"User: Translate this from French to English:\n{example['src']}\nAssistant: {example['tgt']}"
    tokens = tokenizer(
        prompt,
        max_length=max_length,
        truncation=True,
        padding="max_length",
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(
    preprocess,
    batched=False,
    remove_columns=dataset["train"].column_names,
)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama32-1b-qlora-translation",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=3,
    logging_steps=50,
    eval_steps=200,
    save_steps=200,
    save_total_limit=2,
    bf16=True,
    gradient_checkpointing=True,
    lr_scheduler_type="cosine",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    args=training_args,
)

trainer.train()